# Unit 11 — Diffusion sampling: reverse steps, skipped timesteps, executed actions

**Your result:** the code that turns noise into robot actions. Your reverse step equals the `diffusers` DDPM step at every timestep (including the final step with no noise and the jumps of a shortened schedule). Your sampler then produces the actions of a *real* policy object that you can run in the simulator.

Five short exercises (10–20 minutes each) and a larger experiment (30–60 minutes). These times are estimates, not measurements. Full answers are in [the reference notebook](11-diffusion-sampling-reference.ipynb).

**Sources.** [DDPM](https://arxiv.org/abs/2006.11239): equation 7 (posterior mean and variance), Algorithm 2 (sampling), equation 15 (the clean-sample estimate). [Diffusion Policy](https://diffusion-policy.cs.columbia.edu/) for which actions are executed. Pinned code (revision `e0d50211ef236143ae867228662b7dfaba554f02`): `DiffusionModel.conditional_sample`, `DiffusionModel.generate_actions` and `DiffusionPolicy.select_action`; the numerical step lives in `diffusers.DDPMScheduler.step`, which this unit uses **only as an oracle in checks**, never in your answers.

**Where this unit sits.** Units 9–10 are available as package versions (the forward process, the trained denoiser classes). The five functions of this unit are wired into a real policy wrapper through hooks; the wrapper never calls `scheduler.step`.

In [ ]:
# Supplied: workspace, offline caches, imports
from pathlib import Path
import os
import sys


def find_root():
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (path / "manifests" / "act_references.yaml").is_file():
            return path
    raise FileNotFoundError("Start Jupyter inside the repository tree.")


WORKSPACE = find_root()
if str(WORKSPACE / "src") not in sys.path:
    sys.path.insert(0, str(WORKSPACE / "src"))

# Set cache paths and offline mode before any data or model library is imported.
os.environ["HF_HUB_CACHE"] = str(WORKSPACE / ".local-data/hf/hub")
os.environ["HF_DATASETS_CACHE"] = str(WORKSPACE / ".local-data/hf/datasets")
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ.setdefault("TORCH_HOME", str(WORKSPACE / ".local-data/torch"))
CONFIGS, DATA = WORKSPACE / "configs", WORKSPACE / ".local-data"
import math
import time
import torch
from torch import nn
import matplotlib.pyplot as plt

import vla_lab.teaching.diffusion as ref  # package reference: used by checks only
from vla_lab.teaching.diffusion import (  # earlier units, as package versions
    CallbackDiffusionPolicy, ConditionalUnet1D, DDPMSchedule, DiffusionDenoiser, LeRobotDenoiser,
    cosine_betas, cumulative_alphas, epsilon_loss_fn, flat_state_condition, forward_noise,
    load_prepared_diffusion, load_state_action_windows, make_step_noises, take_batch,
)
from vla_lab.teaching.training import masked_action_l1, train_loop
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler

torch.manual_seed(0)
print("PyTorch:", torch.__version__)
print("Workspace:", WORKSPACE)

In [ ]:
# Supplied: a helper that the checks use
def close(actual, expected, message, atol=1e-5):
    """Compare two tensors and say what failed."""
    assert actual.shape == expected.shape, f"{message}: shape {tuple(actual.shape)}, expected {tuple(expected.shape)}"
    worst = (actual.detach() - expected.detach()).abs().max().item()
    assert worst <= atol, f"{message}: largest difference {worst:.2e} is above {atol:.0e}"

## Background: running the process backwards

Training learned `ε_θ(x_t, t)`. Sampling starts from pure noise `x_T ~ N(0, I)` and repeats one **reverse step** until `x_0`:

1. Estimate the clean window: `x̂_0 = (x_t − √(1−ᾱ_t) · ε_θ) / √ᾱ_t`, then **clip** it to `[-1, 1]` (the pinned recipe has `clip_sample=True`; actions are scaled to `[-1, 1]`).
2. Take the posterior mean `μ = c₀ · x̂_0 + c_t · x_t`, with coefficients from the *adjacent* or *skipped* step.
3. Add Gaussian noise with the posterior variance, **except at the final step**.

**Shortened schedules.** The pinned policy trains with `T = 100` steps but samples with `n = 20` steps: `t = 95, 90, …, 5, 0`. A jump from `t` to `prev < t − 1` has no single `β`. Use the ratio `α_t = ᾱ_t / ᾱ_prev` (the product of the skipped alphas) and `β_t = 1 − α_t`.

**Which actions are executed.** The network predicts a window of 16 actions that starts *one frame before* the current observation (`n_obs_steps = 2`). The policy executes indices `n_obs_steps − 1 … n_obs_steps − 1 + n_action_steps − 1` = `1 … 8`, then samples again.

## Exercise 11.1 — Posterior coefficients for adjacent and skipped steps (about 20 min)

**Purpose.** Derive the weights of the reverse step. This is where a shortened schedule differs from the textbook formula: the one-step `β_t` no longer applies.

**Contract (shapes, units, masks).**

- `posterior_coefficients(abar_t, abar_prev)` takes the cumulative alphas at the current step and at the step we jump to (python floats or 0-d tensors). `abar_prev = 1` means "the clean sample" (used for the final step).
- Compute `alpha_t = abar_t / abar_prev` and `beta_t = 1 - alpha_t`.
- Return three 0-d `float32` tensors `(coef_x0, coef_xt, variance)`:
  `coef_x0 = sqrt(abar_prev) * beta_t / (1 - abar_t)`;
  `coef_xt = sqrt(alpha_t) * (1 - abar_prev) / (1 - abar_t)`;
  `variance = (1 - abar_prev) / (1 - abar_t) * beta_t`, clamped to at least `1e-20`.
- For `abar_prev = 1` the result is `coef_x0 = 1`, `coef_xt = 0` (the mean is the clean estimate).

**Method and source.** Posterior `q(x_prev | x_t, x_0) = N(μ̃, β̃ I)` with `μ̃ = (√ᾱ_prev β_t / (1−ᾱ_t)) x_0 + (√α_t (1−ᾱ_prev) / (1−ᾱ_t)) x_t` — [DDPM](https://arxiv.org/abs/2006.11239), equation 7. Skipped steps: apply the same equation to the coarser chain whose one-step alpha is `ᾱ_t / ᾱ_prev`. Why this works: the marginals `q(x_t | x_0)` are the same Gaussians whatever steps we skip, so the chain over the chosen steps has per-jump alpha `ᾱ_t / ᾱ_prev`.

**Pointers.** Write the ratio and `beta_t` first. Then the three formulas in the order of the contract. Use `torch.as_tensor(..., dtype=torch.float32)` so floats and tensors both work. Derive for yourself why `alpha_t * abar_prev = abar_t`.

<details><summary>Detailed hint</summary>

```python
abar_t = torch.as_tensor(abar_t, dtype=torch.float32)
abar_prev = torch.as_tensor(abar_prev, dtype=torch.float32)
alpha_t = abar_t / abar_prev
beta_t = 1.0 - alpha_t
coef_x0 = abar_prev.sqrt() * beta_t / (1.0 - abar_t)
coef_xt = alpha_t.sqrt() * (1.0 - abar_prev) / (1.0 - abar_t)
variance = ((1.0 - abar_prev) / (1.0 - abar_t) * beta_t).clamp_min(1e-20)
return coef_x0, coef_xt, variance
```

</details>

*This is the reference notebook: the next cell is a complete solution.*

In [ ]:
# Reference solution for Exercise 11.1
from __future__ import annotations

def posterior_coefficients(
    abar_t: Tensor | float, abar_prev: Tensor | float
) -> tuple[Tensor, Tensor, Tensor]:
    """Mean coefficients and variance of ``q(x_prev | x_t, x_0)`` (DDPM equations 6 and 7).

    For a jump from ``t`` to any earlier step ``prev`` use the ratio
    ``alpha_t = abar_t / abar_prev`` and ``beta_t = 1 - alpha_t``. (When ``prev = t - 1`` this is
    the ordinary ``1 - beta_t``; for skipped steps it is the product of the skipped alphas.)
    Returns ``(coef_x0, coef_xt, variance)``::

        coef_x0 = sqrt(abar_prev) * beta_t / (1 - abar_t)
        coef_xt = sqrt(alpha_t) * (1 - abar_prev) / (1 - abar_t)
        variance = (1 - abar_prev) / (1 - abar_t) * beta_t     (clamped to at least 1e-20)
    """
    abar_t = torch.as_tensor(abar_t, dtype=torch.float32)
    abar_prev = torch.as_tensor(abar_prev, dtype=torch.float32)
    alpha_t = abar_t / abar_prev
    beta_t = 1.0 - alpha_t
    coef_x0 = abar_prev.sqrt() * beta_t / (1.0 - abar_t)
    coef_xt = alpha_t.sqrt() * (1.0 - abar_prev) / (1.0 - abar_t)
    variance = ((1.0 - abar_prev) / (1.0 - abar_t) * beta_t).clamp_min(1e-20)
    return coef_x0, coef_xt, variance

In [ ]:
# Check: Exercise 11.1
scheduler = DDPMScheduler(num_train_timesteps=100, beta_schedule="squaredcos_cap_v2")
abar, betas = scheduler.alphas_cumprod, scheduler.betas
for t, prev in ((50, 49), (50, 45), (99, 94), (5, 0)):
    c0, ct, var = posterior_coefficients(abar[t], abar[prev])
    assert c0.shape == () and ct.shape == () and var.shape == (), "Return three scalar tensors."
    product = torch.prod(1 - betas[prev + 1 : t + 1])               # the skipped alphas multiplied
    close(abar[t] / abar[prev], product, f"alpha_t for the jump {t} -> {prev}", atol=1e-6)
    beta = 1 - product
    close(c0, abar[prev].sqrt() * beta / (1 - abar[t]), f"coef_x0 for {t} -> {prev}", atol=1e-6)
    close(ct, product.sqrt() * (1 - abar[prev]) / (1 - abar[t]), f"coef_xt for {t} -> {prev}", atol=1e-6)
    close(var, (1 - abar[prev]) / (1 - abar[t]) * beta, f"variance for {t} -> {prev}", atol=1e-7)
c0, ct, var = posterior_coefficients(abar[0], 1.0)
close(c0, torch.tensor(1.0), "final step: coef_x0 is 1", atol=1e-6)
close(ct, torch.tensor(0.0), "final step: coef_xt is 0", atol=1e-6)
assert 0 < float(var) <= 1e-19, "Clamp the variance to at least 1e-20 (it is 0 at the final step)."
c0b, _, _ = posterior_coefficients(float(abar[50]), float(abar[40]))
close(c0b, posterior_coefficients(abar[50], abar[40])[0], "python floats must work too", atol=1e-7)
print("Exercise 11.1 passed")

**Use your function.** Plot how the weights change along a 20-step schedule and a 100-step schedule.

In [ ]:
# Use your function: Exercise 11.1
abar = cumulative_alphas(cosine_betas(100))
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for steps, style in ((100, "-"), (20, "o-")):
    ts = ref.inference_timesteps(100, steps)
    rows = [posterior_coefficients(abar[t], abar[ts[i + 1]] if i + 1 < len(ts) else 1.0) for i, t in enumerate(ts)]
    for axis, index, name in zip(axes, range(3), ("coef_x0 (weight on the clean estimate)", "coef_xt (weight on x_t)", "std of the added noise")):
        values = [float(r[index]) for r in rows]
        axis.plot(ts, [v ** 0.5 if index == 2 else v for v in values], style, label=f"{steps} steps"); axis.set_title(name, fontsize=9); axis.set_xlabel("t")
axes[0].legend(); plt.tight_layout(); plt.show()

## Exercise 11.2 — One reverse step (with the final step) (about 20 min)

**Purpose.** Write the single update that the sampler repeats. The final step is special: there is no noise and the result is the clipped clean estimate.

**Contract (shapes, units, masks).**

- `ddpm_reverse_step(x_t, epsilon, t, prev_t, alpha_bar, noise=None, clip_range=1.0)`.
- `x_t`, `epsilon`, optional `noise`: `(B, H, A)`. `t`, `prev_t`: python ints; `prev_t < 0` means "the final step" (after it there is no earlier step). `alpha_bar`: `(T,)` cumulative alphas.
- `x0_hat = (x_t - sqrt(1 - abar_t) * epsilon) / sqrt(abar_t)`, then clamp to `[-clip_range, clip_range]` if `clip_range` is not `None`.
- `abar_prev = alpha_bar[prev_t]`, or exactly `1` when `prev_t < 0`. Get coefficients from your Exercise 11.1 function. `mean = coef_x0 * x0_hat + coef_xt * x_t`.
- If `prev_t >= 0`: return `mean + sqrt(variance) * noise`; raise `ValueError` if `noise` is `None`. If `prev_t < 0`: return `mean` and **never** use `noise` (it may be `None` or any value).
- Use only the passed tensors. **Do not call a scheduler.**

**Method and source.** DDPM Algorithm 2, line 4: `x_{t-1} = μ̃(x_t, x̂_0) + σ_t z`, with `z = 0` at the last step; `x̂_0` from equation 15 and clipping as in the reference implementation. This is what `DDPMScheduler.step` computes with `prediction_type='epsilon'`.

**Pointers.** Estimate `x0_hat`, clip, fetch `abar_prev` (watch the final-step convention), call `posterior_coefficients`, combine. Draw no random numbers inside this function: the caller passes the noise.

<details><summary>Detailed hint</summary>

```python
abar_t = alpha_bar[t].to(x_t.dtype)
abar_prev = alpha_bar[prev_t].to(x_t.dtype) if prev_t >= 0 else torch.ones((), dtype=x_t.dtype)
x0_hat = (x_t - (1.0 - abar_t).sqrt() * epsilon) / abar_t.sqrt()
if clip_range is not None:
    x0_hat = x0_hat.clamp(-clip_range, clip_range)
coef_x0, coef_xt, variance = posterior_coefficients(abar_t, abar_prev)
mean = coef_x0.to(x_t.dtype) * x0_hat + coef_xt.to(x_t.dtype) * x_t
if prev_t < 0:
    return mean
if noise is None:
    raise ValueError("noise is required for every step except the final one")
return mean + variance.sqrt().to(x_t.dtype) * noise
```

</details>

*This is the reference notebook: the next cell is a complete solution.*

In [ ]:
# Reference solution for Exercise 11.2
from __future__ import annotations

def ddpm_reverse_step(
    x_t: Tensor,
    epsilon: Tensor,
    t: int,
    prev_t: int,
    alpha_bar: Tensor,
    noise: Tensor | None = None,
    clip_range: float | None = 1.0,
) -> Tensor:
    """One reverse step ``x_t -> x_prev`` of ancestral sampling (DDPM Algorithm 2, generalized).

    1. ``x0_hat = (x_t - sqrt(1 - abar_t) * epsilon) / sqrt(abar_t)``; clamp it to
       ``[-clip_range, clip_range]`` when ``clip_range`` is not ``None``.
    2. ``abar_prev = alpha_bar[prev_t]``, or exactly ``1`` when ``prev_t < 0`` (the final step).
    3. ``mean = coef_x0 * x0_hat + coef_xt * x_t`` from ``posterior_coefficients``.
    4. If ``prev_t >= 0`` return ``mean + sqrt(variance) * noise``. At the final step
       (``prev_t < 0``) return ``mean`` with no noise: that equals the clipped ``x0_hat``.
    ``noise`` is required unless this is the final step. ``t`` and ``prev_t`` are Python ints.
    """
    abar_t = alpha_bar[t].to(x_t.dtype)
    abar_prev = alpha_bar[prev_t].to(x_t.dtype) if prev_t >= 0 else torch.ones((), dtype=x_t.dtype)
    x0_hat = (x_t - (1.0 - abar_t).sqrt() * epsilon) / abar_t.sqrt()
    if clip_range is not None:
        x0_hat = x0_hat.clamp(-clip_range, clip_range)
    coef_x0, coef_xt, variance = posterior_coefficients(abar_t, abar_prev)
    mean = coef_x0.to(x_t.dtype) * x0_hat + coef_xt.to(x_t.dtype) * x_t
    if prev_t < 0:
        return mean
    if noise is None:
        raise ValueError("noise is required for every step except the final one")
    return mean + variance.sqrt().to(x_t.dtype) * noise

In [ ]:
# Check: Exercise 11.2
for steps in (5, 20, 100):
    scheduler = DDPMScheduler(num_train_timesteps=100, beta_schedule="squaredcos_cap_v2", clip_sample=True, clip_sample_range=1.0)
    scheduler.set_timesteps(steps)
    times = scheduler.timesteps.tolist()
    abar = scheduler.alphas_cumprod
    g = torch.Generator().manual_seed(0)
    for index, t in enumerate(times):
        prev = times[index + 1] if index + 1 < len(times) else -1
        x = torch.randn(2, 16, 7, generator=g) * 1.5
        eps = torch.randn(2, 16, 7, generator=g)
        noise = torch.randn(2, 16, 7, generator=torch.Generator().manual_seed(100 + index))
        expected = scheduler.step(eps, t, x, generator=torch.Generator().manual_seed(100 + index)).prev_sample   # oracle only
        mine = ddpm_reverse_step(x, eps, t, prev, abar, noise if prev >= 0 else None, 1.0)
        close(mine, expected, f"{steps}-step schedule, t={t} -> prev={prev} differs from diffusers", atol=1e-5)
        if prev < 0:
            x0 = ((x - (1 - abar[t]).sqrt() * eps) / abar[t].sqrt()).clamp(-1, 1)
            close(mine, x0, "the final step returns the clipped clean estimate", atol=1e-5)
            close(ddpm_reverse_step(x, eps, t, prev, abar, torch.full_like(x, 99.0), 1.0), mine, "the final step must ignore any noise", atol=0)
abar = DDPMSchedule.cosine(100).alpha_bar
try:
    ddpm_reverse_step(torch.zeros(1, 2, 2), torch.zeros(1, 2, 2), 10, 5, abar, None)
except ValueError:
    pass
else:
    raise AssertionError("A non-final step without noise must raise ValueError.")
unclipped = ddpm_reverse_step(torch.full((1, 2, 2), 3.0), torch.zeros(1, 2, 2), 0, -1, abar, None, clip_range=None)
clipped = ddpm_reverse_step(torch.full((1, 2, 2), 3.0), torch.zeros(1, 2, 2), 0, -1, abar, None, clip_range=1.0)
assert float(unclipped.max()) > 1.5 and float(clipped.max()) <= 1.0 + 1e-6, "clip_range=None must not clamp; 1.0 must."
print("Exercise 11.2 passed")

**Use your function.** Check the noise your step adds: with a fixed `x_t` and a fixed `ε`, many different noises should spread the result by exactly `sqrt(variance)`.

In [ ]:
# Use your function: Exercise 11.2
abar = cumulative_alphas(cosine_betas(100))
x_t = torch.zeros(4000, 1, 1)
eps = torch.zeros(4000, 1, 1)
noise = torch.randn(4000, 1, 1, generator=torch.Generator().manual_seed(0))
samples = ddpm_reverse_step(x_t, eps, 50, 45, abar, noise)
expected_std = float(posterior_coefficients(abar[50], abar[45])[2].sqrt())
print("empirical spread:", float(samples.std()), "| predicted sqrt(variance):", expected_std)

## Exercise 11.3 — Timesteps of a shortened schedule (about 10 min)

**Purpose.** Choose which of the 100 training steps the sampler visits. Fewer steps are faster; the choice must keep the last step at 0.

**Contract (shapes, units, masks).**

- `inference_timesteps(num_train_timesteps, num_inference_steps)` returns a Python `list` of ints in **descending** order.
- With `ratio = num_train_timesteps // num_inference_steps` the list is `[(n-1)*ratio, …, 2*ratio, ratio, 0]` (length `n`). For `T=100, n=20` that is `[95, 90, …, 5, 0]`.
- The last entry is always `0`. Raise `ValueError` if `n < 1` or `n > T`.

**Method and source.** Evenly spaced ('leading') timesteps as in `diffusers` `DDPMScheduler.set_timesteps`; the pinned policy samples 20 of its 100 steps.

**Pointers.** Make the ascending multiples of the ratio, then reverse them. Validate the inputs first.

<details><summary>Detailed hint</summary>

```python
if not 1 <= num_inference_steps <= num_train_timesteps:
    raise ValueError("num_inference_steps must be between 1 and the training step count")
ratio = num_train_timesteps // num_inference_steps
return [index * ratio for index in reversed(range(num_inference_steps))]
```

</details>

*This is the reference notebook: the next cell is a complete solution.*

In [ ]:
# Reference solution for Exercise 11.3
from __future__ import annotations

def inference_timesteps(num_train_timesteps: int, num_inference_steps: int) -> list[int]:
    """Evenly spaced timesteps in descending order, with spacing ``T // n`` ("leading").

    ``[(n - 1) * r, ..., r, 0]`` with ``r = T // n``. The last timestep is always ``0``.
    """
    if not 1 <= num_inference_steps <= num_train_timesteps:
        raise ValueError("num_inference_steps must be between 1 and the training step count")
    ratio = num_train_timesteps // num_inference_steps
    return [index * ratio for index in reversed(range(num_inference_steps))]

In [ ]:
# Check: Exercise 11.3
for steps in (1, 5, 10, 20, 50, 100):
    scheduler = DDPMScheduler(num_train_timesteps=100, beta_schedule="squaredcos_cap_v2")
    scheduler.set_timesteps(steps)
    mine = inference_timesteps(100, steps)
    assert isinstance(mine, list) and mine == scheduler.timesteps.tolist(), f"{steps} steps: expected {scheduler.timesteps.tolist()[:4]}…, got {mine[:4]}…"
    assert mine[-1] == 0 and mine == sorted(mine, reverse=True), "Descending, ending at 0."
assert inference_timesteps(100, 20)[:3] == [95, 90, 85]
for bad in (0, 101):
    try:
        inference_timesteps(100, bad)
    except ValueError:
        pass
    else:
        raise AssertionError(f"{bad} steps must raise ValueError.")
print("Exercise 11.3 passed")

**Use your function.** How much signal does each step budget leave at its first step? The first timestep decides how far the sampler starts from pure noise.

In [ ]:
# Use your function: Exercise 11.3
abar = cumulative_alphas(cosine_betas(100))
for steps in (2, 5, 10, 20, 100):
    ts = inference_timesteps(100, steps)
    print(f"{steps:>3} steps: first t = {ts[0]:>2}, alpha_bar there = {float(abar[ts[0]]):.4f}, visited: {ts[:4]}…")

## Exercise 11.4 — The reverse loop with explicit noise (about 20 min)

**Purpose.** Run the process from noise to actions. All randomness is passed in, so the same inputs always give the same actions.

**Contract (shapes, units, masks).**

- `sample_actions(denoise, initial_noise, schedule, num_inference_steps, step_noises, clip_range=1.0)`.
- `denoise(x_t, t_batch) -> epsilon` where `t_batch` is a **`long` tensor `(B,)` filled with the current timestep** (same device as `x_t`). `initial_noise (B, H, A)`. `schedule` has `.alpha_bar` and `.num_train_timesteps`.
- `step_noises` is a sequence of `(B, H, A)` tensors: `step_noises[i]` is the noise for step `i`. There must be exactly `num_inference_steps - 1` of them (the final step uses none); otherwise raise `ValueError`.
- Use `inference_timesteps` (Exercise 11.3). Start from `x = initial_noise`. For each timestep: call `denoise(x, t_batch)`, then `x = ddpm_reverse_step(...)` (Exercise 11.2). **Carry the new `x` into the next step.** The previous timestep of the last entry is `-1`.
- Return the final `x`. Do not call a scheduler and do not draw random numbers.

**Method and source.** DDPM Algorithm 2 with the shortened timesteps of Exercise 11.3. The pinned `conditional_sample` is the same loop with `noise_scheduler.step` inside.

**Pointers.** Compute the list of timesteps once. Loop with `enumerate` so that you know the next timestep. Build `t_batch` with `torch.full((batch,), t, dtype=torch.long, device=x.device)`.

<details><summary>Detailed hint</summary>

```python
timesteps = inference_timesteps(schedule.num_train_timesteps, num_inference_steps)
if len(step_noises) != len(timesteps) - 1:
    raise ValueError("need one noise tensor for every step except the last")
x = initial_noise
for index, t in enumerate(timesteps):
    prev_t = timesteps[index + 1] if index + 1 < len(timesteps) else -1
    t_batch = torch.full((x.shape[0],), t, dtype=torch.long, device=x.device)
    epsilon = denoise(x, t_batch)
    noise = step_noises[index] if prev_t >= 0 else None
    x = ddpm_reverse_step(x, epsilon, t, prev_t, schedule.alpha_bar, noise, clip_range)
return x
```

</details>

*This is the reference notebook: the next cell is a complete solution.*

In [ ]:
# Reference solution for Exercise 11.4
from __future__ import annotations

def sample_actions(
    denoise: Callable[[Tensor, Tensor], Tensor],
    initial_noise: Tensor,
    schedule: DDPMSchedule,
    num_inference_steps: int,
    step_noises: Sequence[Tensor],
    clip_range: float | None = 1.0,
) -> Tensor:
    """Run the full reverse process. Returns the final sample ``(B, H, A)``.

    ``denoise(x_t, t_batch)`` returns the predicted noise; ``t_batch`` is a ``long`` ``(B,)``
    tensor filled with the current timestep. Start from ``initial_noise``. For each timestep
    of ``inference_timesteps`` carry the result of one step into the next. The previous
    timestep of the last entry is ``-1``. ``step_noises[i]`` is used by step ``i``; the final
    step uses none. All randomness is passed in, so the same arguments give the same actions.
    """
    timesteps = inference_timesteps(schedule.num_train_timesteps, num_inference_steps)
    if len(step_noises) != len(timesteps) - 1:
        raise ValueError("need one noise tensor for every step except the last")
    x = initial_noise
    for index, t in enumerate(timesteps):
        prev_t = timesteps[index + 1] if index + 1 < len(timesteps) else -1
        t_batch = torch.full((x.shape[0],), t, dtype=torch.long, device=x.device)
        epsilon = denoise(x, t_batch)
        noise = step_noises[index] if prev_t >= 0 else None
        x = ddpm_reverse_step(x, epsilon, t, prev_t, schedule.alpha_bar, noise, clip_range)
    return x

In [ ]:
# Check: Exercise 11.4
schedule = DDPMSchedule.cosine(100)
torch.manual_seed(8)
net = nn.Linear(7, 7)
toy = lambda x, t: net(x) * (t.float().reshape(-1, 1, 1) / 100.0)

# (1) Same noise as the diffusers loop gives the same actions.
for steps in (3, 20, 100):
    scheduler = DDPMScheduler(num_train_timesteps=100, beta_schedule="squaredcos_cap_v2", clip_sample=True, clip_sample_range=1.0)
    scheduler.set_timesteps(steps)
    g = torch.Generator().manual_seed(1)
    initial = torch.randn(2, 16, 7, generator=g)
    expected = initial
    for t in scheduler.timesteps:                                       # oracle loop (uses scheduler.step)
        expected = scheduler.step(toy(expected, torch.full((2,), int(t))), int(t), expected, generator=g).prev_sample
    g2 = torch.Generator().manual_seed(1)
    initial2 = torch.randn(2, 16, 7, generator=g2)
    noises = make_step_noises(steps, initial2.shape, g2)
    close(sample_actions(toy, initial2, schedule, steps, noises, 1.0), expected, f"{steps} steps differ from the diffusers loop", atol=1e-4)

# (2) Behaviour: a perfect denoiser recovers the clean window exactly, whatever the noise.
clean = torch.rand(2, 16, 7) * 1.6 - 0.8
def oracle(x, t):
    ab = schedule.alpha_bar[t].reshape(-1, 1, 1)
    return (x - ab.sqrt() * clean) / (1 - ab).sqrt()
seen = []
def spy(x, t):
    seen.append((x.clone(), t.clone()))
    return oracle(x, t)
start = torch.randn(2, 16, 7, generator=torch.Generator().manual_seed(2))
out = sample_actions(spy, start, schedule, 10, make_step_noises(10, start.shape, torch.Generator().manual_seed(3)))
close(out, clean, "a perfect denoiser must recover the clean window", atol=1e-4)
assert [int(t[0]) for _, t in seen] == [90, 80, 70, 60, 50, 40, 30, 20, 10, 0], "Visit inference_timesteps(100, 10) in order."
assert all(t.dtype == torch.long and t.shape == (2,) for _, t in seen), "t_batch must be a long tensor of shape (B,)."
assert torch.equal(seen[0][0], start) and not torch.equal(seen[1][0], start), "Carry each new sample into the next step."
# (3) Repeatable and strict about the noise count.
noises = make_step_noises(10, start.shape, torch.Generator().manual_seed(3))
assert torch.equal(sample_actions(toy, start, schedule, 10, noises), sample_actions(toy, start, schedule, 10, noises))
try:
    sample_actions(toy, start, schedule, 10, noises[:-1])
except ValueError:
    pass
else:
    raise AssertionError("A wrong number of step noises must raise ValueError.")
print("Exercise 11.4 passed")

**Use your function.** Sample from a *perfect* denoiser built from a real action window (it knows the answer), for several step budgets. The sampler must return the real window each time; with a learned denoiser (the experiment) it only approximates it.

In [ ]:
# Use your function: Exercise 11.4
VALID_REAL = load_state_action_windows(CONFIGS, DATA, "bowl_plate", "validation")
truth = VALID_REAL["action"][100:104]
schedule = DDPMSchedule.cosine(100)
def perfect(x, t):
    ab = schedule.alpha_bar[t].reshape(-1, 1, 1)
    return (x - ab.sqrt() * truth) / (1 - ab).sqrt()
for steps in (1, 5, 20, 100):
    start = torch.randn(truth.shape, generator=torch.Generator().manual_seed(0))
    out = sample_actions(perfect, start, schedule, steps, make_step_noises(steps, start.shape, torch.Generator().manual_seed(1)))
    print(f"{steps:>3} steps: largest error to the real window = {float((out - truth).abs().max()):.1e}")

## Exercise 11.5 — Which actions are executed (about 10 min)

**Purpose.** The network predicts 16 actions, but the robot executes only 8 of them, starting from the one that belongs to the *current* observation. Off by one here means acting on the past.

**Contract (shapes, units, masks).**

- `executed_action_slice(chunk, n_obs_steps, n_action_steps)`: `chunk (B, H, A)` is the sampled window; it starts `n_obs_steps - 1` steps before the current observation.
- Return `chunk[:, start:end]` with `start = n_obs_steps - 1` and `end = start + n_action_steps`.
- For `n_obs_steps=2`, `n_action_steps=8` the returned indices are `1, 2, …, 8` (index 8 included, 9 excluded). The result has `n_action_steps` rows: `(B, n_action_steps, A)`.
- Raise `ValueError` if `n_obs_steps < 1`, `n_action_steps < 1` or `end > H`.

**Method and source.** [Diffusion Policy](https://diffusion-policy.cs.columbia.edu/) (observation horizon, prediction horizon, action horizon) and the table in the docstring of the pinned `DiffusionPolicy.select_action`; the pinned `generate_actions` uses `start = n_obs_steps - 1`, `end = start + n_action_steps`.

**Pointers.** Two numbers, one slice. Test it with a chunk whose values are their own indices.

<details><summary>Detailed hint</summary>

```python
start = n_obs_steps - 1
end = start + n_action_steps
if n_obs_steps < 1 or n_action_steps < 1 or end > chunk.shape[1]:
    raise ValueError("the executed slice must fit inside the sampled window")
return chunk[:, start:end]
```

</details>

*This is the reference notebook: the next cell is a complete solution.*

In [ ]:
# Reference solution for Exercise 11.5
from __future__ import annotations

def executed_action_slice(chunk: Tensor, n_obs_steps: int, n_action_steps: int) -> Tensor:
    """Pick the actions that are executed from a sampled window ``(B, H, A)``.

    The window starts ``n_obs_steps - 1`` steps before the current frame, so the current frame's
    action is at index ``n_obs_steps - 1``. Return indices
    ``n_obs_steps - 1 .. n_obs_steps - 1 + n_action_steps - 1`` (end exclusive at
    ``n_obs_steps - 1 + n_action_steps``). For ``n_obs_steps=2, n_action_steps=8`` that is 1 to 8.
    """
    start = n_obs_steps - 1
    end = start + n_action_steps
    if n_obs_steps < 1 or n_action_steps < 1 or end > chunk.shape[1]:
        raise ValueError("the executed slice must fit inside the sampled window")
    return chunk[:, start:end]

In [ ]:
# Check: Exercise 11.5
chunk = torch.arange(16, dtype=torch.float32).reshape(1, 16, 1).expand(2, 16, 3).clone()
taken = executed_action_slice(chunk, 2, 8)
assert taken.shape == (2, 8, 3), "Expected (B, n_action_steps, A)."
assert taken[0, :, 0].tolist() == [float(i) for i in range(1, 9)], f"For 2/16/8 the indices must be 1..8; got {taken[0, :, 0].tolist()}."
assert executed_action_slice(chunk, 1, 4)[0, :, 0].tolist() == [0.0, 1.0, 2.0, 3.0], "With one observation the window starts at the current action."
assert executed_action_slice(chunk, 2, 15)[0, -1, 0] == 15.0, "The last allowed slice ends at index H-1."
for bad in ((2, 16), (0, 4), (2, 0)):
    try:
        executed_action_slice(chunk, *bad)
    except ValueError:
        pass
    else:
        raise AssertionError(f"{bad} must raise ValueError.")
print("Exercise 11.5 passed")

**Use your function.** The timeline of one chunk, in real units. The dataset is recorded at 10 Hz; the simulator controller runs at 20 Hz. The reference policy sends each action for one controller step and does **not** resample; that behaviour is retained here and is a known mismatch.

In [ ]:
# Use your function: Exercise 11.5
horizon, n_obs, n_act = 16, 2, 8
chunk = torch.arange(horizon).reshape(1, horizon, 1).float()
picked = executed_action_slice(chunk, n_obs, n_act)[0, :, 0].int().tolist()
print("window index        :", list(range(horizon)))
print("observed (history)  :", list(range(n_obs)), "<- index", n_obs - 1, "is the current frame")
print("executed            :", picked)
print(f"recorded demonstration time of the window: {horizon / 10:.1f} s at 10 Hz; executed actions: {n_act / 10:.1f} s of demonstration time")
print(f"as sent to the 20 Hz controller: {n_act / 20:.2f} s of simulated time per chunk (not resampled)")

## Experiment — how many sampling steps are enough? (30–60 min)

**Question.** The pinned policy trains with 100 steps and samples with 20. How does the number of sampling steps change the accuracy of the executed actions and the time per chunk, with the noise held fixed?

**Setup (real data, quick CPU version).** A small state-conditioned denoiser is trained once, here, with the Unit 9 loss (300 updates on real windows; this is *learner trained weights*, not the prepared policy). Then **your** `sample_actions`, `ddpm_reverse_step` and `executed_action_slice` turn fixed noise into action chunks for 256 validation windows. The opt-in sections below run the *same functions* inside the real prepared policy.

**Fixed for every trial.** The trained denoiser; the 256 validation windows; initial noise from seed `11` and the step noises from the same generator; `clip_range` as listed. The 100-step chunk with the same initial noise is the reference for `l1_to_100`.

**Allowed changes (per trial).** `steps` (any of `1, 2, 5, 10, 20, 50, 100`) and `clip` (`True`: clip the clean estimate to `[-1, 1]`; `False`: no clipping).

**Measures.** `l1_to_demo`: masked L1 between the executed actions (indices `1…8`) and the demonstrated actions at those steps. `l1_to_100`: L1 between the executed actions and those of the 100-step sample. `ms_batch1`: milliseconds for one chunk at batch size 1 (the simulator case; noisy, compare within one session). Padded demonstration steps are excluded.

**Prediction first.** In the next cell: at what step budget will `l1_to_demo` stop improving, and what happens with one step?

**Keep failed trials.** Every trial is appended to `TRIALS`. A budget that is much worse is still a result.

In [ ]:
# Larger training experiment: run only when you opt in.
RUN_EXPERIMENT = False

In [ ]:
if RUN_EXPERIMENT:
    PREDICTION = "TODO: replace this text with your prediction."
    print(PREDICTION)

In [ ]:
if RUN_EXPERIMENT:
    # Supplied: trial table helper
    import time
    import pandas as pd

    def show_trials(trials, columns):
        """Print every trial that was run, including failed ones, and return a table."""
        table = pd.DataFrame([{**t["config"], **{c: t.get(c) for c in columns}, "status": t["status"]} for t in trials])
        print(table.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
        return table

In [ ]:
if RUN_EXPERIMENT:
    # Supplied: real windows and a briefly trained small denoiser (learner trained weights)
    TRAIN = load_state_action_windows(CONFIGS, DATA, "bowl_plate", "train")
    VALID = load_state_action_windows(CONFIGS, DATA, "bowl_plate", "validation")
    SCHEDULE = DDPMSchedule.cosine(100)

    torch.manual_seed(0)
    SMALL = DiffusionDenoiser(ConditionalUnet1D(7, 16, (32, 64, 128), step_embed_dim=32), SCHEDULE, flat_state_condition)
    draws = torch.Generator().manual_seed(1)
    training_batches = [take_batch(TRAIN, torch.randint(0, len(TRAIN["action"]), (64,), generator=draws)) for _ in range(300)]
    started = time.perf_counter()
    train_loop(SMALL, training_batches, torch.optim.Adam(SMALL.parameters(), lr=1e-3, betas=(0.95, 0.999)), epsilon_loss_fn, steps=300)
    SMALL.eval()
    print(f"trained 300 updates in {time.perf_counter() - started:.0f} s")

    pick = torch.linspace(0, len(VALID["action"]) - 1, 256).round().long()
    EVAL = take_batch(VALID, pick)
    CONDITION = SMALL.condition_fn(EVAL)
    denoise = lambda x, t: SMALL.unet(x, t, global_cond=CONDITION)
    DEMO, DEMO_PAD = EVAL["action"][:, 1:9], EVAL["action_is_pad"][:, 1:9]    # the executed indices of the demonstration window

In [ ]:
if RUN_EXPERIMENT:
    # Supplied: harness. Your sampler, reverse step and slice are in the real path.
    SEED_NOISE, N_OBS, N_ACT = 11, 2, 8
    SHAPE = EVAL["action"].shape
    BASELINE = {"steps": 20, "clip": True}
    TRIALS = []


    def sample_chunk(steps, clip, batch_size=None):
        rows = SHAPE[0] if batch_size is None else batch_size
        g = torch.Generator().manual_seed(SEED_NOISE)
        initial = torch.randn(SHAPE, generator=g)[:rows]
        noises = [n[:rows] for n in make_step_noises(steps, SHAPE, g)]
        cond = CONDITION[:rows]
        started = time.perf_counter()
        with torch.no_grad():
            window = sample_actions(lambda x, t: SMALL.unet(x, t, global_cond=cond), initial, SCHEDULE, steps, noises, 1.0 if clip else None)
        return window, time.perf_counter() - started


    REFERENCE_WINDOW, _ = sample_chunk(100, True)
    REFERENCE_EXECUTED = executed_action_slice(REFERENCE_WINDOW, N_OBS, N_ACT)


    def run_trial(config):
        window, _ = sample_chunk(config["steps"], config["clip"])
        executed = executed_action_slice(window, N_OBS, N_ACT)
        timings = [sample_chunk(config["steps"], config["clip"], batch_size=1)[1] for _ in range(5)]
        return {
            "l1_to_demo": masked_action_l1(executed, DEMO, DEMO_PAD).item(),
            "l1_to_100": (executed - REFERENCE_EXECUTED).abs().mean().item(),
            "ms_batch1": 1000 * sorted(timings)[2],
        }


    def record(config):
        config = {**BASELINE, **config}
        try:
            TRIALS.append({"config": config, "status": "ok", **run_trial(config)})
        except Exception as error:  # a failed trial is evidence too
            TRIALS.append({"config": config, "status": f"failed: {type(error).__name__}: {error}"})


    record(BASELINE)
    COLUMNS = ["l1_to_demo", "l1_to_100", "ms_batch1"]
    show_trials(TRIALS, COLUMNS);

**Your trials.** Add at least three trials. Changing `steps` is the main question; try `clip=False` once. Budget: 300 updates per trial.

In [ ]:
if RUN_EXPERIMENT:
    # Reference choice of trials: one change per trial.
    MY_TRIALS = [{"steps": 1}, {"steps": 2}, {"steps": 5}, {"steps": 10}, {"steps": 50}, {"steps": 100}, {"clip": False}]
    for trial in MY_TRIALS:
        record(trial)
    show_trials(TRIALS, COLUMNS);

In [ ]:
if RUN_EXPERIMENT:
    # Check: the protocol was followed.
    assert TRIALS[0]["config"] == BASELINE, "The first trial must be the baseline."
    assert len(TRIALS) >= 4, "Run the baseline and at least three trials of your own."
    assert all(set(t["config"]) == set(BASELINE) for t in TRIALS), "Only steps and clip may change."
    assert all(t["config"]["steps"] in (1, 2, 5, 10, 20, 50, 100) for t in TRIALS)
    ok = [t for t in TRIALS if t["status"] == "ok"]
    assert all(torch.isfinite(torch.tensor([t["l1_to_demo"], t["l1_to_100"], t["ms_batch1"]])).all() for t in ok), "Measures must be finite."
    print(f"{len(TRIALS)} trials kept ({len(ok)} ok). Fixed noise seed {SEED_NOISE}; {SHAPE[0]} validation windows; denoiser trained once for 300 updates.")

In [ ]:
if RUN_EXPERIMENT:
    # Use your function: accuracy against time for every trial
    ok = [t for t in TRIALS if t["status"] == "ok"]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
    for t in ok:
        label = f"{t['config']['steps']}" + ("" if t["config"]["clip"] else " (no clip)")
        axes[0].scatter(t["ms_batch1"], t["l1_to_demo"]); axes[0].annotate(label, (t["ms_batch1"], t["l1_to_demo"]), fontsize=8)
        axes[1].scatter(t["ms_batch1"], t["l1_to_100"]); axes[1].annotate(label, (t["ms_batch1"], t["l1_to_100"]), fontsize=8)
    axes[0].set_xlabel("ms per chunk (batch 1)"); axes[0].set_ylabel("l1_to_demo")
    axes[1].set_xlabel("ms per chunk (batch 1)"); axes[1].set_ylabel("l1_to_100")
    plt.tight_layout(); plt.show()

**Explain (2–4 sentences, edit this cell).** Which budget would you choose for a robot, and why? How does the distance to the 100-step sample differ from the distance to the demonstration, and which one measures task success? What does `clip=False` change? Remember that this denoiser sees only the state (no images) and was trained for 300 updates.

*Optional extension:* repeat the baseline and two other budgets with another `SEED_NOISE`; report the spread. Keep all rows.

## Opt-in A: your sampler inside the real prepared policy (offline, CPU)

`CallbackDiffusionPolicy` runs the **prepared** diffusion policy's image encoders and U-Net, but its sampling loop is **your** `sample_actions` (with your `ddpm_reverse_step`) and its action choice is **your** `executed_action_slice`; it never calls `scheduler.step`. This section samples chunks for 12 real validation frames with the same fixed noise for every budget, and compares them with the demonstrated actions and with the 100-step sample. It reads local files only. The workload is 12 validation windows and up to 100 sampling steps per window. Set `RUN_REAL_POLICY_OFFLINE = True` to run it.

In [ ]:
RUN_REAL_POLICY_OFFLINE = False
REAL_SEED_NOISE = 11
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
STEP_BUDGETS = (5, 10, 20, 50, 100)

if RUN_REAL_POLICY_OFFLINE:
    import json
    import pandas as pd
    from torch.utils.data import default_collate
    from vla_lab.teaching.act_model import open_libero_dataset
    from vla_lab.teaching.diffusion import stack_observation
    from vla_lab.training.diffusion import FPS, diffusion_delta_timestamps

    checkpoint = DATA / "runs/diffusion-bowl-plate-reference-2000/checkpoint-002000"
    if not (checkpoint / "model.safetensors").is_file():
        raise FileNotFoundError(f"Prepared diffusion checkpoint missing: {checkpoint}")
    policy, preprocessor, _ = load_prepared_diffusion(checkpoint, device=DEVICE)
    config = policy.config
    dataset = open_libero_dataset(CONFIGS, DATA, "bowl_plate", "validation", diffusion_delta_timestamps(config, FPS))
    frames = torch.linspace(0, len(dataset) - 1, 12).round().long().tolist()
    batch = preprocessor(default_collate([dataset[i] for i in frames]))
    observation = stack_observation(policy, batch)
    wrapper = CallbackDiffusionPolicy(policy, sample_actions, executed_action_slice, seed=REAL_SEED_NOISE)
    real_demo = batch["action"][:, config.n_obs_steps - 1 : config.n_obs_steps - 1 + config.n_action_steps]
    real_pad = batch["action_is_pad"][:, config.n_obs_steps - 1 : config.n_obs_steps - 1 + config.n_action_steps]
    shape = (len(frames), config.horizon, 7)
    windows, rows = {}, []
    for steps in STEP_BUDGETS:
        g = torch.Generator(device=DEVICE).manual_seed(REAL_SEED_NOISE)
        initial = torch.randn(shape, generator=g, device=DEVICE)
        noises = make_step_noises(steps, shape, g, device=DEVICE)
        started = time.perf_counter()
        windows[steps] = wrapper.predict_window(observation, initial, noises, num_inference_steps=steps)
        rows.append({"steps": steps, "seconds_for_12_chunks": time.perf_counter() - started})
    reference = executed_action_slice(windows[max(STEP_BUDGETS)], config.n_obs_steps, config.n_action_steps)
    for row in rows:
        executed = executed_action_slice(windows[row["steps"]], config.n_obs_steps, config.n_action_steps)
        row["l1_to_demo"] = masked_action_l1(executed, real_demo, real_pad).item()
        row["l1_to_max_steps"] = (executed - reference).abs().mean().item()
    print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.4g}"))
    saved = DATA.parent / "artifacts/reference/diffusion/sampler_ablation.json"
    if saved.is_file():
        ablation = json.loads(saved.read_text())["sampling_steps"]
        print("Prepared policy results (saved by the project: simulator, development states 0-4, RTX 3070; NOT produced here):")
        for key, value in ablation.items():
            print(f"  {key:>3} steps: {value['successes']}/{value['attempts']} successes, mean sampler time {value['weighted_mean_sampler_ms']:.0f} ms")
else:
    print("Offline real-policy comparison is off. Set RUN_REAL_POLICY_OFFLINE=True to run the 12-window comparison.")

## Opt-in B: closed-loop rollouts on development states 10–14 (simulator)

This runs the prepared policy with **your sampler** in the LIBERO simulator through the project's `run_policy_episode`, on the *methods* development states `10–14` only (states `0–9` are for exploration, `15–19` for the intervention unit, and the final states `20–49` are never used in these units). Reset seed `20260930 + state` and the policy noise seed are fixed; every attempt, including failures and errors, is kept in `ATTEMPTS`. Each episode has at most 300 simulator steps (the controller runs at 20 Hz; the demonstrations were recorded at 10 Hz and the policy is **not** resampled). The episode budget is 300 simulator steps per attempt. The code selects CUDA when available and otherwise uses CPU; it loads one model only. Set `RUN_DEV_STATE_ROLLOUTS = True` to run it.

In [ ]:
RUN_DEV_STATE_ROLLOUTS = False
DEV_STATES = (10, 11, 12, 13, 14)     # methods states; narrow this list to run fewer episodes
ROLLOUT_BUDGETS = (5, 20)             # sampler steps to compare
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_STEPS = 300
RESET_BASE_SEED = 20260930
ATTEMPTS = []

if RUN_DEV_STATE_ROLLOUTS:
    import pandas as pd
    from vla_lab.sim.episode import open_libero_task, run_policy_episode

    assert all(10 <= s <= 14 for s in DEV_STATES), "Use development states 10-14 only in this unit."
    checkpoint = DATA / "runs/diffusion-bowl-plate-reference-2000/checkpoint-002000"
    if not (checkpoint / "model.safetensors").is_file():
        raise FileNotFoundError(f"Prepared diffusion checkpoint missing: {checkpoint}")
    policy, preprocessor, postprocessor = load_prepared_diffusion(checkpoint, device=DEVICE)
    for steps in ROLLOUT_BUDGETS:
        for state_id in DEV_STATES:
            wrapper = CallbackDiffusionPolicy(policy, sample_actions, executed_action_slice,
                                              num_inference_steps=steps, seed=RESET_BASE_SEED + state_id)
            environment = None
            try:
                environment = open_libero_task("bowl_plate", state_id, CONFIGS, DATA)
                result = run_policy_episode(environment, wrapper, preprocessor, postprocessor, "bowl_plate", state_id, CONFIGS,
                                            partition="development", max_steps=MAX_STEPS, reset_seed=RESET_BASE_SEED + state_id)
                ATTEMPTS.append({"steps": steps, "state": state_id, "success": result.success, "env_steps": result.steps,
                                 "ended": result.termination_reason, "chunks": wrapper.chunks_made,
                                 "mean_sampler_ms": 1000 * sum(wrapper.sample_seconds) / max(len(wrapper.sample_seconds), 1)})
            except Exception as error:  # keep failed attempts
                ATTEMPTS.append({"steps": steps, "state": state_id, "success": False, "ended": f"error: {type(error).__name__}: {error}"})
            finally:
                if environment is not None:
                    try:
                        environment.close()
                    except Exception as error:
                        ATTEMPTS.append({"steps": steps, "state": state_id, "success": False,
                                         "ended": f"close error: {type(error).__name__}: {error}"})
    table = pd.DataFrame(ATTEMPTS)
    print(table.to_string(index=False, float_format=lambda v: f"{v:.1f}"))
    print(table.groupby("steps")["success"].agg(["sum", "count"]).rename(columns={"sum": "successes", "count": "attempts"}))
else:
    print("Closed-loop rollouts are off. Set RUN_DEV_STATE_ROLLOUTS=True to run", len(DEV_STATES) * len(ROLLOUT_BUDGETS), "episodes.")

## What you built

- `posterior_coefficients`, `ddpm_reverse_step`, `inference_timesteps`, `sample_actions` and `executed_action_slice`: DDPM sampling that equals `diffusers` at every timestep (final and skipped steps included) without calling it, and a slice that picks the right actions.
- A real policy object whose actions come from your sampler, with fixed noise, step budgets, timing and every trial or attempt kept.

**Next:** Unit 12A opens a vision-language-action policy (SmolVLA): inputs and trainable parameters. Unit 12B replaces diffusion by flow matching.